In [3]:
# --- Sample data for testing (delete before committing) ---
rng = np.random.default_rng(42)
n_obs, omega, alpha, beta, true_nu = 2500, 0.05, 0.08, 0.90, 5

z = rng.standard_t(true_nu, n_obs) / np.sqrt(true_nu / (true_nu - 2))
returns = np.empty(n_obs)
sigma2 = omega / (1 - alpha - beta)
eps = 0.0
for i in range(n_obs):
    sigma2 = omega + alpha * eps**2 + beta * sigma2
    eps = np.sqrt(sigma2) * z[i]
    returns[i] = eps

sample_data = pd.Series(returns,
                       index=pd.bdate_range("2016-01-01", periods=n_obs),
                       name="Return_Pct")

In [2]:
import numpy as np
import pandas as pd
from arch import arch_model


def garch_dist_model(data, p=1, q=1, dist="normal", horizon=1):

    returns = data.dropna()

    model = arch_model(returns, mean="Constant", vol="GARCH", p=p, q=q, dist=dist)

    result = model.fit(disp="off")

    forecast = result.forecast(horizon=horizon)
    vol = np.sqrt(forecast.variance.iloc[-1])
    vol.index = pd.bdate_range(returns.index[-1] + pd.Timedelta(days=1), periods=horizon)
    vol.name = "Forecast_Vol_Pct"

    print(vol.to_string())

    return result

In [4]:
garch_dist_model(sample_data,1,1,"normal",1)

2025-08-01    1.30697
Freq: B


                     Constant Mean - GARCH Model Results                      
Dep. Variable:             Return_Pct   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -4327.38
Distribution:                  Normal   AIC:                           8662.77
Method:            Maximum Likelihood   BIC:                           8686.07
                                        No. Observations:                 2500
Date:                Thu, Sep 24 2026   Df Residuals:                     2499
Time:                        22:55:03   Df Model:                            1
                                  Mean Model                                 
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
mu            -0.0171  2.500e-02     -0.684      0.494 

In [ ]:
garch_dist_model(sample_data,1,1,"t",1)

In [ ]:
garch_dist_model(sample_data,1,1,"GED",1)